# Ejemplo aplicado: regresión lineal — Diabetes (progresión)

Misma plantilla que [01-regresion-lineal.ipynb](01-regresion-lineal.ipynb), ya configurada para `diabetes.csv` (target `disease_progression`, separador `,`). Target: progresión de la enfermedad (continuo). Incluye faltantes en algunas columnas (el imputer del pipeline los gestiona).

> Para un CSV nuevo, parte de la **plantilla genérica** `01-regresion-lineal.ipynb`, no de este archivo.


In [ ]:
# =============================================================================
# Helpers — funciones reutilizables (misma lógica en todo el benchmark)
# =============================================================================
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer  # pipeline distinto por tipo de columna
from sklearn.impute import SimpleImputer  # rellenar NaN antes de escalar/codificar
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline  # encadena: preprocesado → modelo
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Las tres funciones siguientes se usan en varios pasos del notebook:
#   1. infer_feature_columns  → arma la lista de columnas X (predictores)
#   2. infer_column_types     → separa X en numéricas y categóricas
#   3. build_preprocess       → crea el ColumnTransformer (imputer + scaler / one-hot)



## 1. Explorar el CSV (antes de CONFIG)

Ejecuta esta celda con un CSV nuevo **solo** para ver columnas, tipos y faltantes. Con eso rellenas `TARGET_COL`, `DROP_COLS`, etc. en la celda siguiente.

Si el `head()` se ve mal (todo en una columna), cambia `PREVIEW_SEP` a `","`, `";"` o `"\t"`.

In [ ]:
# Ruta y separador del CSV (único ajuste antes de ver la tabla)
PREVIEW_PATH = "data/diabetes.csv"
PREVIEW_SEP = ","  # Prueba: ","  |  ";"  |  "\t"

df_preview = pd.read_csv(PREVIEW_PATH, sep=PREVIEW_SEP)

print(f"Filas: {len(df_preview):,}  |  Columnas: {len(df_preview.columns)}")
print("\n--- Nombres de columnas (índice : nombre) ---")
for i, col in enumerate(df_preview.columns):
    print(f"  {i:2d}: {col!r}")

print("\n--- Tipos de datos (dtypes) ---")
print(df_preview.dtypes)

print("\n--- Primeras filas ---")
display(df_preview.head())

print("\n--- Valores faltantes por columna ---")
missing = df_preview.isna().sum()
if missing.any():
    display(missing[missing > 0].to_frame("nulos"))
else:
    print("No hay valores faltantes.")

# Sugerencia para NUMERIC_COLS / CATEGORICAL_COLS en CONFIG (si dejas None, se infieren igual)
_num = df_preview.select_dtypes(include=[np.number]).columns.tolist()
_cat = df_preview.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
print("\n--- Sugerencia automática de tipos ---")
print("Numéricas (int/float):", _num)
print("Categóricas (object/category/bool/string):", _cat)

print(
    "\n>>> Siguiente paso: en CONFIG usa el mismo PREVIEW_PATH → DATA_PATH, "
    "PREVIEW_SEP → CSV_SEP, y elige TARGET_COL (columna numérica a predecir)."
)

## 2. CONFIG

In [ ]:
# ========== CONFIG (editar al cambiar de dataset) ==========
# Mismos valores que PREVIEW_PATH y PREVIEW_SEP en la celda de exploración
DATA_PATH = "data/diabetes.csv"
# Ejemplo otro archivo: DATA_PATH = "data/wine_quality_red.csv"

CSV_SEP = ","
# Ejemplos: "," (estándar)  |  ";" (wine quality)  |  "\t" (TSV)

# Columna que quieres PREDECIR (consumo en millas por galón)
TARGET_COL = "disease_progression"
# Progresión de la diabetes (variable continua a predecir)

# Columnas que NO deben usarse como features (ids, texto libre, duplicados del target)
DROP_COLS = []
# sex (1/2) es numérico en el CSV; para tratarlo como categórica: CATEGORICAL_COLS = ["sex"]
# Ejemplos: ["id", "id_vivienda"]  |  ["PassengerId", "Name"] (Titanic, si el target es Survived)

# Si None: todas las columnas salvo TARGET_COL y DROP_COLS
FEATURE_COLS = None
# Ejemplo manual: FEATURE_COLS = ["alcohol", "volatile acidity", "sulphates"]

# Si None: se infieren por tipo (int/float vs object/category)
NUMERIC_COLS = None
# Ejemplo: NUMERIC_COLS = ["edad", "ingresos", "hijos"]

CATEGORICAL_COLS = None
# Ejemplo: CATEGORICAL_COLS = ["ciudad", "tipo_contrato", "color"]

# Fracción para test (0.2 = 20 % test, 80 % train)
TEST_SIZE = 0.2

# Semilla para reproducir el mismo split y modelos aleatorios
RANDOM_STATE = 42

# Métrica para ordenar la tabla de comparación (mayor R² = mejor en regresión)
METRIC_PRINCIPAL = "r2"


def build_models():
    """Diccionario nombre → estimador. Comenta líneas para excluir modelos del benchmark."""
    from sklearn.ensemble import (
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.linear_model import Lasso, LinearRegression, Ridge
    from xgboost import XGBRegressor
    from catboost import CatBoostRegressor

    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=5000),
        "RandomForest": RandomForestRegressor(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        "XGBoost": XGBRegressor(
            random_state=RANDOM_STATE, verbosity=0, n_estimators=100, n_jobs=-1
        ),
        "CatBoost": CatBoostRegressor(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }
    return models


MODELS = build_models()


## 3. Carga de datos

In [ ]:
df = pd.read_csv(DATA_PATH, sep=CSV_SEP)
print("Shape:", df.shape)
df.head()

## 4. Calidad de datos

In [ ]:
print(df.info())
print("\nFaltantes por columna:")
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else "Sin valores faltantes")

## 5. Visualización rápida

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET_COL].hist(ax=axes[0], bins=20, edgecolor="black")
axes[0].set_title(f"Distribución de {TARGET_COL}")
num_feat = df.select_dtypes(include=[np.number]).columns.drop(TARGET_COL, errors="ignore")
if len(num_feat):
    feat = num_feat[0]
    axes[1].scatter(df[feat], df[TARGET_COL], alpha=0.4)
    axes[1].set_xlabel(feat)
    axes[1].set_ylabel(TARGET_COL)
plt.tight_layout()
plt.show()

## 6. Separar X / y y split train-test

In [ ]:
feature_cols = infer_feature_columns(df, TARGET_COL, DROP_COLS, FEATURE_COLS)
X = df[feature_cols]
y = df[TARGET_COL]
numeric_cols, categorical_cols = infer_column_types(X, NUMERIC_COLS, CATEGORICAL_COLS)
print("Numéricas:", numeric_cols)
print("Categóricas:", categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

## 7. Preprocesado (compartido por todos los modelos)

In [ ]:
preprocess = build_preprocess(numeric_cols, categorical_cols)
preprocess

## 8. Comparar modelos

In [ ]:
def regression_metrics(y_true, y_pred):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}


def evaluate_models(models, preprocess, X_train, X_test, y_train, y_test):
    rows = []
    for name, estimator in models.items():
        pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        rows.append({"modelo": name, **regression_metrics(y_test, y_pred)})
    return pd.DataFrame(rows).sort_values(METRIC_PRINCIPAL, ascending=False)

results = evaluate_models(MODELS, preprocess, X_train, X_test, y_train, y_test)
display(results.round(4))

ax = results.plot(x="modelo", y=METRIC_PRINCIPAL, kind="barh", legend=False, figsize=(8, 5))
ax.set_xlabel("R² (test)")
ax.set_title("Comparación de modelos — regresión")
plt.tight_layout()
plt.show()


## 9. Detalle del mejor modelo

In [ ]:
best_name = results.iloc[0]["modelo"]
print(f"Mejor modelo (test): {best_name}")

best_est = MODELS[best_name]
best_pipe = Pipeline([("preprocess", preprocess), ("model", best_est)])
best_pipe.fit(X_train, y_train)
y_pred_best = best_pipe.predict(X_test)

# --- Scatter real vs predicho ---
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_best, alpha=0.5, edgecolors="k", linewidths=0.3)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlabel("Valor real")
ax.set_ylabel("Predicción")
ax.set_title(f"{best_name}: real vs predicho (test)")
plt.tight_layout()
plt.show()

# --- Matriz de confusión (target discreto o binned) ---
from sklearn.metrics import ConfusionMatrixDisplay

y_true_arr = np.asarray(y_test)
y_pred_arr = np.asarray(y_pred_best)

if pd.Series(y_train).nunique() <= 25:
    y_true_cm = np.round(y_true_arr).astype(int)
    y_pred_cm = np.round(y_pred_arr).astype(int)
    cm_note = "valores redondeados"
else:
    n_bins = 5
    bin_edges = np.unique(np.quantile(y_train, np.linspace(0, 1, n_bins + 1)))
    if len(bin_edges) < 2:
        bin_edges = np.linspace(float(np.min(y_train)), float(np.max(y_train)), n_bins + 1)
    y_true_cm = np.digitize(y_true_arr, bin_edges[1:-1])
    y_pred_cm = np.digitize(y_pred_arr, bin_edges[1:-1])
    cm_note = f"{len(bin_edges) - 1} intervalos (cuantiles de train)"

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_true_cm, y_pred_cm, ax=ax)
ax.set_title(f"Matriz de confusión ({cm_note}) — {best_name}")
plt.tight_layout()
plt.show()



## Checklist: nuevo dataset

1. Coloca el CSV en `data/` y actualiza `DATA_PATH`, `TARGET_COL` y `DROP_COLS`.
2. Revisa faltantes y `dtypes` (celdas de EDA).
3. Ajusta `NUMERIC_COLS` / `CATEGORICAL_COLS` si la detección automática falla.
4. En clasificación, comprueba el balance de clases.
5. Opcional: edita `build_models()` para añadir o quitar algoritmos.
6. Ejecuta todas las celdas y compara la tabla de resultados.
